In [ ]:
# Example 1: Basic Profiling with Context Manager
import pyinstrument
import time
import random

def slow_function():
    """Simulate slow computation for profiling demonstration"""
    time.sleep(0.1)
    # Simulate some computational work
    result = sum(random.random() * i for i in range(10000))
    return result

def another_slow_function():
    """Another function with different performance characteristics"""
    time.sleep(0.05)
    # Different computation pattern
    data = [random.randint(1, 100) for _ in range(5000)]
    return sorted(data)

# Basic profiling using context manager
profiler = pyinstrument.Profiler()

print("Starting profiling session...")
profiler.start()

# Code to profile
result1 = slow_function()
result2 = another_slow_function()
result3 = slow_function()

profiler.stop()

# Get profiling results
print("Profiling completed. Results:")
print(f"Total execution time: {profiler.last_session.duration:.3f} seconds")

# Display simplified profiling output
output = profiler.output_text(unicode=False, color=False)
print("\nProfile Summary (first 500 chars):")
print(output[:500] + "..." if len(output) > 500 else output)

print(f"\nExecution results: {result1:.2f}, {len(result2)} items, {result3:.2f}")

In [ ]:
# Example 2: Advanced Profiling with Decorators and Filtering
import pyinstrument
import time
import json

@pyinstrument.profile
def cpu_intensive_task():
    """CPU-intensive task for profiling"""
    # Simulate complex calculation
    total = 0
    for i in range(100000):
        total += i ** 2 + (i * 3.14159) / 2.71828
    return total

def io_intensive_task():
    """I/O intensive task simulation"""
    time.sleep(0.02)  # Simulate I/O wait
    return "I/O completed"

def mixed_workload():
    """Mixed CPU and I/O workload"""
    data = []
    for i in range(5):
        # CPU work
        result = sum(x**2 for x in range(1000))
        data.append(result)
        # I/O simulation
        time.sleep(0.01)
    return data

# Advanced profiling with custom configuration
profiler = pyinstrument.Profiler(interval=0.001)  # 1ms sampling interval

print("Advanced profiling with custom configuration...")
with profiler:
    cpu_result = cpu_intensive_task()
    io_result = io_intensive_task()
    mixed_result = mixed_workload()

# Analyze profiling results with filtering
session = profiler.last_session
print(f"Session duration: {session.duration:.3f} seconds")
print(f"Sample count: {session.sample_count}")

# Get filtered output focusing on our functions
output = profiler.output_text(unicode=False, color=False)
lines = output.split('\n')
relevant_lines = [line for line in lines if any(func in line for func in 
                 ['cpu_intensive_task', 'io_intensive_task', 'mixed_workload'])]

print("\nFunction-specific profiling results:")
for line in relevant_lines[:10]:  # Show first 10 relevant lines
    print(line)

print(f"\nResults summary: CPU={cpu_result:.0f}, IO='{io_result}', Mixed={len(mixed_result)} items")

In [ ]:
# Example 3: Asynchronous Code Profiling and Analysis
import pyinstrument
import asyncio
import time
import random

async def async_api_call(delay=0.05):
    """Simulate async API call"""
    await asyncio.sleep(delay)
    return {"status": "success", "data": random.randint(1, 100)}

async def async_database_query(query_time=0.03):
    """Simulate async database operation"""
    await asyncio.sleep(query_time)
    return [{"id": i, "value": random.random()} for i in range(10)]

async def cpu_bound_async_task():
    """CPU-bound work in async context"""
    # Some CPU work that doesn't yield
    result = sum(i ** 2 for i in range(50000))
    await asyncio.sleep(0.001)  # Small yield
    return result

async def async_workflow():
    """Complex async workflow combining multiple operations"""
    # Parallel API calls
    api_tasks = [async_api_call(0.02) for _ in range(3)]
    api_results = await asyncio.gather(*api_tasks)
    
    # Database query
    db_result = await async_database_query(0.03)
    
    # CPU work
    cpu_result = await cpu_bound_async_task()
    
    return {
        "api_calls": len(api_results),
        "db_records": len(db_result),
        "cpu_result": cpu_result
    }

# Profile async code - use await directly in Jupyter
profiler = pyinstrument.Profiler(async_mode='enabled')

print("Profiling async workflow...")
profiler.start()

# Run async workflow directly with await (Jupyter has event loop)
result = await async_workflow()

profiler.stop()

print(f"Async workflow completed: {result}")
print(f"Total time: {profiler.last_session.duration:.3f} seconds")

# Show async-aware profiling output
output = profiler.output_text(unicode=False, color=False)
print("\nAsync Profiling Summary (first 400 chars):")
print(output[:400] + "..." if len(output) > 400 else output)

In [ ]:
# Example 4: Performance Comparison and Optimization Analysis
import pyinstrument
import time
import json

def inefficient_sort(data):
    """Inefficient bubble sort implementation"""
    n = len(data)
    for i in range(n):
        for j in range(0, n-i-1):
            if data[j] > data[j+1]:
                data[j], data[j+1] = data[j+1], data[j]
    return data

def efficient_sort(data):
    """Efficient built-in sort"""
    return sorted(data)

def json_processing_slow(data):
    """Inefficient JSON processing"""
    result = []
    for item in data:
        json_str = json.dumps(item)
        parsed = json.loads(json_str)
        result.append(parsed)
    return result

def json_processing_fast(data):
    """More efficient processing"""
    return [item.copy() for item in data]  # Direct copy instead of JSON round-trip

# Performance comparison setup
test_data = [{"id": i, "value": i * 2, "name": f"item_{i}"} for i in range(500)]
sort_data = list(range(100, 0, -1))  # Reverse sorted for worst-case

print("Performance Comparison Analysis")
print("=" * 40)

# Profile inefficient operations
profiler = pyinstrument.Profiler()
profiler.start()

slow_sorted = inefficient_sort(sort_data.copy())
slow_json = json_processing_slow(test_data[:50])  # Smaller dataset for demo

profiler.stop()
slow_duration = profiler.last_session.duration

# Profile efficient operations  
profiler.reset()
profiler.start()

fast_sorted = efficient_sort(sort_data.copy())
fast_json = json_processing_fast(test_data[:50])

profiler.stop()
fast_duration = profiler.last_session.duration

print(f"Slow operations took: {slow_duration:.4f} seconds")
print(f"Fast operations took: {fast_duration:.4f} seconds")
print(f"Performance improvement: {slow_duration/fast_duration:.1f}x faster")

# Analysis output
output = profiler.output_text(unicode=False, color=False)
print("\nOptimized Code Profile (first 300 chars):")
print(output[:300] + "..." if len(output) > 300 else output)

In [ ]:
# Example 5: Production Application Profiling and Monitoring
import pyinstrument
import time
import random
import tempfile
import os

class DataProcessor:
    """Sample data processing application for profiling"""
    
    def __init__(self):
        self.cache = {}
    
    def load_data(self, size=1000):
        """Simulate data loading"""
        time.sleep(0.02)  # I/O simulation
        return [{"id": i, "value": random.random(), "category": f"cat_{i%5}"} 
                for i in range(size)]
    
    def process_item(self, item):
        """Process individual data item"""
        # Simulate CPU work
        processed_value = item["value"] ** 2 + (item["id"] * 0.001)
        return {
            "processed_id": item["id"],
            "result": processed_value,
            "category": item["category"]
        }
    
    def aggregate_by_category(self, processed_data):
        """Aggregate results by category"""
        aggregates = {}
        for item in processed_data:
            cat = item["category"]
            if cat not in aggregates:
                aggregates[cat] = {"count": 0, "sum": 0, "avg": 0}
            aggregates[cat]["count"] += 1
            aggregates[cat]["sum"] += item["result"]
        
        # Calculate averages
        for cat, data in aggregates.items():
            data["avg"] = data["sum"] / data["count"]
        
        return aggregates

def production_workflow():
    """Simulate production data processing workflow"""
    processor = DataProcessor()
    
    # Load data
    raw_data = processor.load_data(500)
    
    # Process data
    processed_data = [processor.process_item(item) for item in raw_data]
    
    # Aggregate results
    aggregates = processor.aggregate_by_category(processed_data)
    
    return {
        "processed_items": len(processed_data),
        "categories": len(aggregates),
        "sample_aggregate": list(aggregates.values())[0] if aggregates else None
    }

# Production profiling with HTML output
print("Production Application Profiling")
print("=" * 35)

profiler = pyinstrument.Profiler(interval=0.001)

print("Running production workflow...")
with profiler:
    result = production_workflow()

print(f"Workflow completed: {result}")
print(f"Total execution time: {profiler.last_session.duration:.3f} seconds")

# Generate comprehensive profiling report
temp_dir = tempfile.gettempdir()
html_file = os.path.join(temp_dir, "profile_report.html")

html_output = profiler.output_html()
print(f"\nGenerated HTML report: {len(html_output)} characters")
print("HTML report contains detailed call stack visualization")

# Text summary for monitoring
text_output = profiler.output_text(unicode=False, color=False)
lines = text_output.split('\n')[:15]  # First 15 lines for summary

print("\nProduction Profiling Summary:")
for line in lines:
    if line.strip():
        print(line)

print(f"\nPerformance metrics ready for monitoring dashboard")